# Lesson 5A: N-Step Methods & Eligibility Traces - Theory

## Introduction

So far we've learned:
- **DP**: Uses full model (not practical)
- **MC**: Uses full returns (high variance)
- **TD(0)**: Uses 1-step bootstrap (low variance but biased)

**N-step methods** interpolate: bootstrap after n steps instead of 1.

## Key Insight

By varying n, we slide between TD and MC:
- n=1: TD(0) (immediate bootstrap)
- n=∞: MC (full trajectory)
- n∈(1,∞): Balanced bias-variance

**Eligibility traces** make this efficient by tracking which states/actions contributed to prediction errors.

## Setup & Core Ideas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print('N-Step TD and Eligibility Traces')

## N-Step Returns

### Definition

**n-step return**:
$$G_t^{(n)} = R_{t+1} + \gamma R_{t+2} + ... + \gamma^{n-1} R_{t+n} + \gamma^n V(S_{t+n})$$

Sum first n rewards, then bootstrap off state at t+n.

### Special Cases

- **n=1**: $G_t^{(1)} = R_{t+1} + \gamma V(S_{t+1})$ (TD)
- **n=∞**: $G_t^{(∞)} = R_{t+1} + \gamma R_{t+2} + ...$ (MC)

### Bias-Variance

- n↑: More actual rewards → lower bias, higher variance
- n↓: More bootstrapping → higher bias, lower variance

## N-Step TD Prediction

### Update Rule

$$V(S_t) \leftarrow V(S_t) + \alpha[G_t^{(n)} - V(S_t)]$$

Waits n steps before updating. Simple but inefficient.

### Disadvantage

Can only update every n steps. Slow in practice.

## Eligibility Traces

### Core Idea

Instead of waiting n steps, **update all visited states immediately** with eligibility weights.

### Forward View

$$e_t(s) = \gamma \lambda e_{t-1}(s) + \mathbb{1}_{s=S_t}$$

- Decay past eligibilities by γλ
- Add 1 when state s is visited

### TD(λ) Update

$$V(s) \leftarrow V(s) + \alpha \delta_t e_t(s)$$ for all s

where $\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$ (TD error)

### Interpretation

- Each visited state gets credit weighted by recency
- λ controls credit decay:
  - λ=0: Only current state credited (TD(0))
  - λ=1: Full credit to all visited states (MC)
  - λ∈(0,1): Geometrically decayed credit

## Forward-Backward View Equivalence

### Forward View

$$V(S_t) \leftarrow V(S_t) + \alpha[G_t^{\lambda} - V(S_t)]$$

where $G_t^{\lambda} = (1-\lambda)\sum_{n=1}^{\infty} \lambda^{n-1} G_t^{(n)}$

Weighted average of all n-step returns.

### Backward View

$$\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$$
$$e_t(s) = \gamma \lambda e_{t-1}(s) + \mathbb{1}_{s=S_t}$$
$$V(s) \leftarrow V(s) + \alpha \delta_t e_t(s)$$ for all s

### The Key Theorem

**Forward and backward views are mathematically equivalent.**

Choosing eligibility traces is just a computational trick to implement n-step returns efficiently.

## Sarsa(λ): On-Policy TD(λ) Control

### Algorithm

```
Initialize e(s,a) = 0 for all s,a

loop:
  S, A ← state, action from ε-greedy
  S', R ← environment step
  
  δ ← R + γQ(S',A') - Q(S,A)
  e(S,A) ← e(S,A) + 1  # accumulating trace
  
  for all s,a:
    Q(s,a) ← Q(s,a) + α δ e(s,a)
    e(s,a) ← γλ e(s,a)  # decay traces
  
  S,A ← S',A'
```

Updates Q immediately with eligibility weighting.

## Summary

### Key Concepts

1. **N-step returns**: Blend n rewards with bootstrap
2. **Eligibility traces**: Track state recency for credit assignment
3. **TD(λ)**: Weighted average of n-step returns via traces
4. **Forward/backward equivalence**: Different view, same algorithm
5. **Sarsa(λ)**: On-policy control with eligibility traces

### Why This Matters

- Interpolates between TD and MC smoothly
- Efficient credit assignment to multiple states per step
- Faster convergence than TD(0) in many domains
- Foundation for modern deep RL (eligibility traces in experience replay)

### Next: Lesson 5B (Practical)

Implement Sarsa(λ) on RandomWalk—see how λ trades off bias and variance.